# Findrr ML-001: Title Expander - training and export (v2.1.0)

Implements `docs/ml/models/title-expander.md`: fine-tune `google/flan-t5-small` to
generate alternative job titles from a title, skills and a seniority level.

**Data:** O*NET 31.0 (public, CC BY 4.0): reported titles, job titles, software skills.
No real user data. Splits are by input title (D-009): every occupation is learned, and test
inputs are titles never used as a training input.

**Run on Kaggle:** Settings -> Accelerator **GPU T4**, **Internet on** -> Run All.
Then download `title_expander_bundle.tar.gz` and extract it into
`models/title-expander/v2.1.0/` in your Findrr folder.

**Export (D-004):** the model ships as ONNX fp32 with a KV cache (`encoder.onnx`,
`decoder.onnx`, `embeddings.npy`), so the Findrr app runs it without PyTorch. The
export is checked against the model's own logits, and **all metrics below are
measured on the exported ONNX files** with the app's own decoding code
(`findrr.ml.t5_onnx_decoding`, embedded below). This evaluation runs on the CPU and
takes about 20-30 minutes.

**No retraining needed for a v2.0.0 model:** upload your accepted
v2.0.0 bundle as one archive (`.tar.gz` or `.zip`) to a private Kaggle
dataset, attach it, and set `REUSE_BUNDLE_DIR` below to the dataset folder. The notebook verifies its checksums and acceptance, skips
training and re-exports it.

Decoding is constrained to real O*NET titles (D-007), and the bundle ships the
title dictionary as `titles.json`. The notebook stops before export if the model
fails its acceptance criteria: occupation precision >= the PROPOSED threshold,
valid title ratio >= 0.85, repetition < 0.05, and >= 3 distinct titles for
"Software Developer". ROUGE-L and BLEU-4 are reported only.

In [ ]:
# Environment report. Reads package metadata only: importing transformers/torch here,
# before the pip install below replaces them, would mix two versions in this kernel.
import platform, sys
from importlib import metadata
print('Python', sys.version.split()[0], '|', platform.platform())
for pkg in ('torch', 'transformers', 'tokenizers', 'datasets', 'sentence-transformers',
            'numpy', 'onnxruntime', 'huggingface-hub'):
    try:
        print(f'{pkg:>22} {metadata.version(pkg)}')
    except metadata.PackageNotFoundError:
        print(f'{pkg:>22} not installed')

In [ ]:
# Pure-Python libraries whose APIs this notebook uses are pinned exactly.
# Compiled packages (torch, tokenizers, safetensors, numpy) come from the Kaggle
# image: pinning old releases forces source builds on newer Python versions.
# sentencepiece is not needed: flan-t5 loads its fast tokenizer from tokenizer.json.
# onnx/onnxruntime get a minimum version so pip can pick a prebuilt wheel (D-004 export).
!pip install -q --prefer-binary transformers==4.46.3 datasets==3.1.0 accelerate==1.1.1 \
    rouge-score==0.1.2 sacrebleu==2.4.3 "onnx>=1.17.0" "onnxruntime>=1.20.1"

In [ ]:
# O*NET database (CC BY 4.0, U.S. Department of Labor / ETA). Public occupational data only;
# no real user data is used anywhere in this notebook.
ONET_VERSION = '31_0'
ONET_URL = f'https://www.onetcenter.org/dl_files/database/db_{ONET_VERSION}_text.zip'
# SHA-256 of db_31_0_text.zip as downloaded on 2026-11-26. Change both values together.
ONET_SHA256 = '6883548adf5fde64cf6f801b35d15519c9225f2732c3cab0e281c652d16b23a9'
# Optional offline copy: attach the extracted folder as a Kaggle dataset at this path.
ONET_OFFLINE_DIR = '/kaggle/input/onet-db'
SEED = 42

In [ ]:
import hashlib, io, os, zipfile, urllib.request
from pathlib import Path

def obtain_onet_dir() -> Path:
    """Return a directory with the O*NET text files; verify the archive checksum."""
    offline = Path(ONET_OFFLINE_DIR)
    if offline.exists():
        hits = list(offline.rglob('Occupation Data.txt'))
        if hits:
            print(f'Using attached O*NET dataset at {hits[0].parent}')
            return hits[0].parent
    print(f'Downloading {ONET_URL} ...')
    with urllib.request.urlopen(ONET_URL, timeout=120) as resp:
        blob = resp.read()
    digest = hashlib.sha256(blob).hexdigest()
    if digest != ONET_SHA256:
        raise RuntimeError(
            f'O*NET archive checksum mismatch: got {digest}, expected {ONET_SHA256}. '
            'Refusing to train on unverified data.'
        )
    target = Path('/kaggle/working/onet')
    with zipfile.ZipFile(io.BytesIO(blob)) as zf:
        zf.extractall(target)
    return next(target.rglob('Occupation Data.txt')).parent

ONET_DIR = obtain_onet_dir()

In [ ]:
import csv, re, random
from collections import defaultdict
from pathlib import Path

def _find(root: Path, *names: str) -> Path:
    """Return the first existing file among names (O*NET renamed files across releases)."""
    for name in names:
        p = Path(root) / name
        if p.exists():
            return p
    raise FileNotFoundError(f'None of {names} found in {root}')

def _rows(path: Path):
    with open(path, encoding='utf-8', newline='') as fh:
        yield from csv.DictReader(fh, delimiter='\t')

_PAREN = re.compile(r'\s*\([^)]*\)')

def clean_title(title: str, short: str = '') -> str:
    """Prefer the O*NET short title; drop parenthetical expansions; collapse whitespace."""
    base = short if short and short.strip().lower() not in ('n/a', '') else title
    base = _PAREN.sub('', base).replace('(', ' ').replace(')', ' ')
    return ' '.join(base.replace('/', ' / ').split()).strip(' ,;')

def norm(title: str) -> str:
    return ' '.join(re.sub(r'[^\w\s]', ' ', title.lower()).split())

def load_onet(root) -> dict:
    root = Path(root)
    occupations = {r['O*NET-SOC Code']: r['Title'] for r in _rows(_find(root, 'Occupation Data.txt'))}
    reported = defaultdict(list)
    for r in _rows(_find(root, 'Sample of Reported Titles.txt')):
        t = clean_title(r['Reported Job Title'])
        if t and t not in reported[r['O*NET-SOC Code']]:
            reported[r['O*NET-SOC Code']].append(t)
    job_titles = defaultdict(list)
    for r in _rows(_find(root, 'Job Titles.txt', 'Alternate Titles.txt')):
        title_col = 'Job Title' if 'Job Title' in r else 'Alternate Title'
        t = clean_title(r[title_col], r.get('Short Title', ''))
        if t and t not in job_titles[r['O*NET-SOC Code']]:
            job_titles[r['O*NET-SOC Code']].append(t)
    skills = defaultdict(list)
    for r in _rows(_find(root, 'Software Skills.txt', 'Technology Skills.txt')):
        ex = r.get('Workplace Example') or r.get('Example') or ''
        if ex:
            skills[r['O*NET-SOC Code']].append((ex.strip(), r.get('Hot Technology', 'N') == 'Y'))
    related = defaultdict(list)
    for r in _rows(_find(root, 'Related Occupations.txt')):
        related[r['O*NET-SOC Code']].append((r['Related O*NET-SOC Code'], r['Relatedness Tier']))
    return {
        'occupations': occupations,
        'reported': dict(reported),
        'job_titles': dict(job_titles),
        'skills': dict(skills),
        'related': dict(related),
    }

def split_by_occupation(socs, seed: int = 42, fractions=(0.8, 0.1, 0.1)) -> dict:
    """Assign whole occupations to train/val/test, stratified by SOC major group.

    Every title of an occupation lands in exactly one split, so evaluation titles are
    never seen in training (spec: disjoint titles, stratified by occupational domain).
    """
    by_group = defaultdict(list)
    for soc in sorted(set(socs)):
        by_group[soc[:2]].append(soc)
    rng = random.Random(seed)
    assignment = {}
    for group in sorted(by_group):
        members = by_group[group]
        rng.shuffle(members)
        n = len(members)
        n_val = max(1, round(n * fractions[1])) if n >= 3 else 0
        n_test = max(1, round(n * fractions[2])) if n >= 3 else 0
        for i, soc in enumerate(members):
            if i < n_test:
                assignment[soc] = 'test'
            elif i < n_test + n_val:
                assignment[soc] = 'val'
            else:
                assignment[soc] = 'train'
    return assignment

In [ ]:
# Copied verbatim from src/findrr/ml/title_constraint.py by
# scripts/generate_kaggle_notebooks.py. Do not edit here.
"""Constrained decoding for ML-001: generate only real titles from a fixed dictionary (D-007).

The model's output format is ``"Title A, Title B, Title C"``. A token trie over the dictionary
restricts every title segment to a known title, separated by the comma token, and a title that
has already been generated cannot be completed again. Invented and repeated titles are therefore
impossible by construction.

This module is pure Python (no torch import) and is used verbatim by both the Findrr runner
and ``notebooks/01_train_title_expander.ipynb`` (the notebook generator embeds this file's
source), so training-time evaluation and app inference use the same constraint.
"""

from collections import Counter
from typing import Dict, Iterable, List, Optional, Sequence, Tuple


class _Node:
    __slots__ = ("children", "terminal", "count")

    def __init__(self) -> None:
        self.children: Dict[int, "_Node"] = {}
        self.terminal = False
        self.count = 0  # number of dictionary titles that end at or below this node


class TitleTrieConstraint:
    """Token-level trie over dictionary titles, usable as ``prefix_allowed_tokens_fn``."""

    def __init__(
        self,
        titles: Iterable[str],
        tokenizer,
        max_titles: Optional[int] = None,
    ) -> None:
        self.comma_id = tokenizer.convert_tokens_to_ids(",")
        self.eos_id = tokenizer.eos_token_id
        self.pad_id = tokenizer.pad_token_id
        self.max_titles = max_titles
        self.root = _Node()
        self.titles: List[str] = []
        self.skipped: List[str] = []
        for title in titles:
            if "," in title:
                self.skipped.append(title)
                continue
            ids = list(tokenizer(title, add_special_tokens=False)["input_ids"])
            with_comma = list(tokenizer(title + ",", add_special_tokens=False)["input_ids"])
            # Skip titles whose last piece merges with the separator (e.g. '3,' in T5):
            # they could never be followed by the comma token in a real output.
            if not ids or with_comma != ids + [self.comma_id]:
                self.skipped.append(title)
                continue
            if self._walk(ids) is not None and self._walk(ids).terminal:
                self.skipped.append(title)  # same token sequence as an earlier title
                continue
            node = self.root
            node.count += 1
            for tok in ids:
                node = node.children.setdefault(tok, _Node())
                node.count += 1
            node.terminal = True
            self.titles.append(title)

    @staticmethod
    def _segments(generated: Sequence[int], comma_id: int) -> List[List[int]]:
        segments: List[List[int]] = [[]]
        for tok in generated:
            if tok == comma_id:
                segments.append([])
            else:
                segments[-1].append(tok)
        return segments

    def _walk(self, ids: Sequence[int]) -> Optional[_Node]:
        node = self.root
        for tok in ids:
            node = node.children.get(tok)
            if node is None:
                return None
        return node

    def allowed_tokens(self, generated: Sequence[int]) -> List[int]:
        """Token IDs allowed next, given the decoder tokens generated so far (no start token)."""
        if self.eos_id in generated:
            return [self.pad_id, self.eos_id]
        segments = self._segments(generated, self.comma_id)
        completed = {tuple(s) for s in segments[:-1]}
        current = segments[-1]
        node = self._walk(current)
        if node is None:
            return [self.eos_id]
        # Allow a branch only if it can still end in a title that has not been used yet, so
        # decoding never spells out a repeat (it would otherwise reach a dead end).
        depth = len(current)
        used_below: Counter = Counter(
            done[depth]
            for done in completed
            if len(done) > depth and list(done[:depth]) == current
        )
        allowed = [
            tok for tok, child in node.children.items() if child.count > used_below[tok]
        ]
        if node.terminal and tuple(current) not in completed:
            if self.max_titles is None or len(segments) < self.max_titles:
                allowed.append(self.comma_id)
            allowed.append(self.eos_id)
        return allowed or [self.eos_id]

    def prefix_allowed_tokens_fn(self, decoder_start_token_id: int):
        """Adapter for ``model.generate(prefix_allowed_tokens_fn=...)``."""

        def fn(batch_id: int, input_ids) -> List[int]:
            ids = input_ids.tolist() if hasattr(input_ids, "tolist") else list(input_ids)
            if ids and ids[0] == decoder_start_token_id:
                ids = ids[1:]
            return self.allowed_tokens(ids)

        return fn

    def complete_titles(self, generated: Sequence[int], tokenizer) -> List[str]:
        """Decode complete, distinct dictionary titles; drop a title cut off by max length."""
        ids = [t for t in generated if t not in (self.pad_id, self.eos_id)]
        out: List[str] = []
        seen: set = set()
        for seg in self._segments(ids, self.comma_id):
            node = self._walk(seg)
            key: Tuple[int, ...] = tuple(seg)
            if seg and node is not None and node.terminal and key not in seen:
                seen.add(key)
                out.append(tokenizer.decode(seg, skip_special_tokens=True).strip())
        return out

In [ ]:
# Copied verbatim from src/findrr/ml/t5_onnx_decoding.py by
# scripts/generate_kaggle_notebooks.py. Do not edit here.
"""ONNX Runtime inference for ML-001 (Flan-T5): constrained beam search with a KV cache (D-004).

Pure NumPy + ONNX Runtime + ``tokenizers``: no torch, no transformers. Used verbatim by both
the Findrr runner and ``notebooks/01_train_title_expander.ipynb`` (the notebook generator embeds
this file's source), so the notebook evaluates exactly what the app runs.

Bundle layout (written by ``t5_onnx_export``):
- ``encoder.onnx``: ``inputs_embeds (B,S,D)``, ``attention_mask (B,S)`` -> ``hidden (B,S,D)``
- ``decoder.onnx``: one decoding step. Inputs ``inputs_embeds (B,1,D)``,
  ``encoder_hidden_states``, ``encoder_attention_mask``, ``past_keys`` and ``past_values``
  ``(L,B,H,P,Dk)``; outputs ``logits (B,V)``, ``present_keys`` and ``present_values``
  ``(L,B,H,P+1,Dk)``.
- ``embeddings.npy``: the shared token embedding (V,D), memory-mapped and stored once.

The beam search mirrors transformers' ``generate(num_beams=N, do_sample=False,
length_penalty=1.0, early_stopping=False, prefix_allowed_tokens_fn=...)``.
"""

from pathlib import Path
from typing import Callable, List, Optional, Sequence, Tuple

import numpy as np


class TokenizersAdapter:
    """The slice of the Hugging Face tokenizer API that ``TitleTrieConstraint`` needs."""

    def __init__(self, tokenizer_json: Path, eos_token_id: int = 1, pad_token_id: int = 0,
                 max_length: int = 128) -> None:
        from tokenizers import Tokenizer

        self.tokenizer = Tokenizer.from_file(str(tokenizer_json))
        self.tokenizer.enable_truncation(max_length=max_length)
        self.tokenizer.no_padding()
        self.eos_token_id = eos_token_id
        self.pad_token_id = pad_token_id

    def convert_tokens_to_ids(self, token: str) -> Optional[int]:
        return self.tokenizer.token_to_id(token)

    def __call__(self, text: str, add_special_tokens: bool = True) -> dict:
        return {"input_ids": self.tokenizer.encode(text, add_special_tokens=add_special_tokens).ids}

    def decode(self, ids: Sequence[int], skip_special_tokens: bool = True) -> str:
        return self.tokenizer.decode(list(ids), skip_special_tokens=skip_special_tokens)


def _log_softmax(x: np.ndarray) -> np.ndarray:
    m = x.max(axis=-1, keepdims=True)
    return x - m - np.log(np.exp(x - m).sum(axis=-1, keepdims=True))


class T5OnnxGenerator:
    """Encoder + KV-cached decoder sessions with constrained beam search."""

    def __init__(self, bundle_dir: Path, threads: int = 4) -> None:
        import onnxruntime as ort

        bundle_dir = Path(bundle_dir)
        options = ort.SessionOptions()
        options.intra_op_num_threads = threads
        providers = ["CPUExecutionProvider"]
        self.encoder = ort.InferenceSession(str(bundle_dir / "encoder.onnx"), options,
                                            providers=providers)
        self.decoder = ort.InferenceSession(str(bundle_dir / "decoder.onnx"), options,
                                            providers=providers)
        # Memory-mapped: only the rows that are looked up are paged in.
        self.embeddings = np.load(bundle_dir / "embeddings.npy", mmap_mode="r")
        past = {i.name: i.shape for i in self.decoder.get_inputs()}["past_keys"]
        self.num_layers, self.num_heads, self.head_dim = int(past[0]), int(past[2]), int(past[4])

    def _embed(self, ids: np.ndarray) -> np.ndarray:
        return np.asarray(self.embeddings[ids], dtype=np.float32)

    def encode(self, input_ids: Sequence[int]) -> Tuple[np.ndarray, np.ndarray]:
        ids = np.array([list(input_ids)], dtype=np.int64)
        mask = np.ones_like(ids)
        hidden = self.encoder.run(None, {"inputs_embeds": self._embed(ids),
                                         "attention_mask": mask})[0]
        return hidden, mask

    def generate(
        self,
        input_ids: Sequence[int],
        allowed_tokens: Callable[[List[int]], List[int]],
        num_beams: int,
        max_new_tokens: int,
        decoder_start_token_id: int = 0,
        eos_token_id: int = 1,
    ) -> List[int]:
        """Best beam's generated token IDs (without the start token, ending in EOS if any)."""
        hidden, mask = self.encode(input_ids)
        nb = num_beams
        hidden = np.repeat(hidden, nb, axis=0)
        mask = np.repeat(mask, nb, axis=0)
        shape = (self.num_layers, nb, self.num_heads, 0, self.head_dim)
        past_k = np.zeros(shape, dtype=np.float32)
        past_v = np.zeros(shape, dtype=np.float32)
        seqs: List[List[int]] = [[decoder_start_token_id] for _ in range(nb)]
        beam_scores = np.array([0.0] + [-1e9] * (nb - 1))
        hyps: List[Tuple[float, List[int]]] = []

        def add_hypothesis(seq: List[int], sum_logprobs: float, generated_len: int) -> None:
            hyps.append((sum_logprobs / generated_len, seq))
            hyps.sort(key=lambda h: -h[0])
            del hyps[nb:]

        finished_by_score = False
        for _ in range(max_new_tokens):
            last = np.array([[s[-1]] for s in seqs], dtype=np.int64)
            logits, past_k, past_v = self.decoder.run(None, {
                "inputs_embeds": self._embed(last),
                "encoder_hidden_states": hidden,
                "encoder_attention_mask": mask,
                "past_keys": past_k,
                "past_values": past_v,
            })
            logprobs = _log_softmax(logits.astype(np.float64))
            masked = np.full_like(logprobs, -np.inf)
            for b in range(nb):
                allowed = allowed_tokens(seqs[b][1:])
                masked[b, allowed] = logprobs[b, allowed]
            scores = (masked + beam_scores[:, None]).reshape(-1)
            vocab = logprobs.shape[1]
            top = np.argsort(-scores, kind="stable")[: 2 * nb]
            cur_len = len(seqs[0]) + 1
            next_seqs: List[List[int]] = []
            next_scores: List[float] = []
            next_rows: List[int] = []
            for rank, idx in enumerate(top):
                beam, token = divmod(int(idx), vocab)
                score = float(scores[idx])
                if not np.isfinite(score):
                    continue
                if token == eos_token_id:
                    if rank < nb:
                        add_hypothesis(seqs[beam] + [token], score, cur_len - 1)
                else:
                    next_seqs.append(seqs[beam] + [token])
                    next_scores.append(score)
                    next_rows.append(beam)
                if len(next_seqs) == nb:
                    break
            best_running = float(scores[top[0]])
            if len(hyps) >= nb and hyps[-1][0] >= best_running / (cur_len - 1):
                finished_by_score = True
                break
            if not next_seqs:
                finished_by_score = True
                break
            while len(next_seqs) < nb:  # keep the batch shape; a dead beam never wins
                next_seqs.append(next_seqs[-1])
                next_scores.append(-1e9)
                next_rows.append(next_rows[-1])
            rows = np.array(next_rows)
            past_k, past_v = past_k[:, rows], past_v[:, rows]
            seqs, beam_scores = next_seqs, np.array(next_scores)
        if not finished_by_score:
            for seq, score in zip(seqs, beam_scores):
                add_hypothesis(seq, float(score), len(seq) - 1)
        return hyps[0][1][1:]

In [ ]:
# Copied verbatim from src/findrr/ml/t5_onnx_export.py by
# scripts/generate_kaggle_notebooks.py. Do not edit here.
"""Export a fine-tuned Flan-T5 (ML-001) to the KV-cached ONNX bundle read by ``t5_onnx_decoding``.

Runs only in ``notebooks/01_train_title_expander.ipynb`` (the notebook generator embeds this
file's source) and in tests. The Findrr app never imports it, so torch stays out of the app.

The decoder step is written directly against the model weights (T5 v1.1: RMSNorm, relative
position bias, gated-GELU feed-forward) instead of tracing transformers' own decoder. The graph
therefore does not depend on the transformers version or its cache classes.
``export_t5_bundle`` checks the exported graphs against the model's own logits and raises if
they differ.
"""

import math
from pathlib import Path


def _build_modules(model):
    """Return (EncoderWrapper, DecoderStep) torch modules for a T5ForConditionalGeneration."""
    import torch

    cfg = model.config
    if not getattr(cfg, "is_gated_act", True) or "gelu" not in getattr(cfg, "dense_act_fn", ""):
        raise ValueError("Only T5 v1.1 / Flan-T5 (gated-gelu) checkpoints are supported.")
    eps = cfg.layer_norm_epsilon
    heads, d_kv = cfg.num_heads, cfg.d_kv
    num_buckets = cfg.relative_attention_num_buckets
    max_distance = cfg.relative_attention_max_distance

    def rms_norm(x, weight):
        variance = x.pow(2).mean(-1, keepdim=True)
        return weight * (x * torch.rsqrt(variance + eps))

    def gelu_new(x):
        return 0.5 * x * (1.0 + torch.tanh(math.sqrt(2.0 / math.pi) * (x + 0.044715 * x.pow(3))))

    def split_heads(x):  # (B,T,H*Dk) -> (B,H,T,Dk)
        b, t = x.shape[0], x.shape[1]
        return x.view(b, t, heads, d_kv).transpose(1, 2)

    class EncoderWrapper(torch.nn.Module):
        def __init__(self, encoder):
            super().__init__()
            self.encoder = encoder

        def forward(self, inputs_embeds, attention_mask):
            return self.encoder(inputs_embeds=inputs_embeds,
                                attention_mask=attention_mask).last_hidden_state

    class DecoderStep(torch.nn.Module):
        def __init__(self, model):
            super().__init__()
            self.blocks = model.decoder.block
            self.final_norm = model.decoder.final_layer_norm
            self.lm_head = model.lm_head
            self.rel_bias = model.decoder.block[0].layer[0].SelfAttention.relative_attention_bias
            # Output scaling depends on the transformers version and how it reads
            # tie_word_embeddings, so export_t5_bundle chooses it by comparison with the model.
            self.scale = 1.0

        def position_bias(self, past_len):
            # Unidirectional buckets for query position == past_len over keys 0..past_len.
            keys = torch.arange(past_len + 1, dtype=torch.long)
            n = past_len - keys  # >= 0
            max_exact = num_buckets // 2
            is_small = n < max_exact
            safe = torch.clamp(n, min=1).float()
            large = max_exact + (
                torch.log(safe / max_exact) / math.log(max_distance / max_exact)
                * (num_buckets - max_exact)
            ).to(torch.long)
            large = torch.clamp(large, max=num_buckets - 1)
            bucket = torch.where(is_small, n, large)
            return self.rel_bias(bucket).permute(1, 0).unsqueeze(0).unsqueeze(2)  # (1,H,1,P+1)

        def forward(self, inputs_embeds, encoder_hidden_states, encoder_attention_mask,
                    past_keys, past_values):
            h = inputs_embeds
            bias = self.position_bias(past_keys.shape[3])
            neg = torch.finfo(h.dtype).min
            cross_mask = (1.0 - encoder_attention_mask[:, None, None, :].to(h.dtype)) * neg
            presents_k, presents_v = [], []
            for i, block in enumerate(self.blocks):
                sa, ca, ff = block.layer[0], block.layer[1], block.layer[2]
                x = rms_norm(h, sa.layer_norm.weight)
                q = split_heads(sa.SelfAttention.q(x))
                k = torch.cat([past_keys[i], split_heads(sa.SelfAttention.k(x))], dim=2)
                v = torch.cat([past_values[i], split_heads(sa.SelfAttention.v(x))], dim=2)
                presents_k.append(k)
                presents_v.append(v)
                attn = torch.softmax(q @ k.transpose(-1, -2) + bias, dim=-1) @ v
                h = h + sa.SelfAttention.o(attn.transpose(1, 2).reshape(h.shape[0], 1, -1))
                x = rms_norm(h, ca.layer_norm.weight)
                q = split_heads(ca.EncDecAttention.q(x))
                k = split_heads(ca.EncDecAttention.k(encoder_hidden_states))
                v = split_heads(ca.EncDecAttention.v(encoder_hidden_states))
                attn = torch.softmax(q @ k.transpose(-1, -2) + cross_mask, dim=-1) @ v
                h = h + ca.EncDecAttention.o(attn.transpose(1, 2).reshape(h.shape[0], 1, -1))
                x = rms_norm(h, ff.layer_norm.weight)
                d = ff.DenseReluDense
                h = h + d.wo(gelu_new(d.wi_0(x)) * d.wi_1(x))
            h = rms_norm(h, self.final_norm.weight) * self.scale
            logits = self.lm_head(h)[:, -1, :]
            return logits, torch.stack(presents_k), torch.stack(presents_v)

    return EncoderWrapper(model.encoder), DecoderStep(model)


def export_t5_bundle(model, out_dir: Path, opset: int = 17, tolerance: float = 1e-3) -> dict:
    """Write encoder.onnx, decoder.onnx and embeddings.npy to ``out_dir``; return parity stats.

    Raises ``RuntimeError`` if the ONNX graphs disagree with the model's own forward pass.
    """
    import numpy as np
    import onnxruntime as ort
    import torch

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    model = model.cpu().eval()
    cfg = model.config
    encoder, step = _build_modules(model)
    emb = model.shared.weight.detach().float()
    np.save(out_dir / "embeddings.npy", emb.numpy())

    vocab = emb.shape[0]
    ids = torch.tensor([[t % vocab for t in (100, 200, 300, 400)] + [cfg.eos_token_id]])
    mask = torch.ones_like(ids)
    layers, heads, d_kv = cfg.num_decoder_layers, cfg.num_heads, cfg.d_kv
    seq = [cfg.decoder_start_token_id] + [t % vocab for t in (37, 1782, 6, 2116, 3, 9)]
    with torch.no_grad():
        hidden = encoder(emb[ids], mask)
        ref = model(input_ids=ids, attention_mask=mask,
                    decoder_input_ids=torch.tensor([seq])).logits[0]
        # Pick the decoder output scaling that reproduces the model's own first-step logits.
        empty = torch.zeros(layers, 1, heads, 0, d_kv)
        for scale in (1.0, cfg.d_model ** -0.5):
            step.scale = scale
            first = step(emb[torch.tensor([[seq[0]]])], hidden, mask, empty, empty)[0][0]
            if float((first - ref[0]).abs().max()) <= tolerance:
                break
        else:
            raise RuntimeError("Decoder step does not reproduce the model's logits.")
        ref = ref.numpy()
        torch.onnx.export(
            encoder, (emb[ids], mask), str(out_dir / "encoder.onnx"),
            input_names=["inputs_embeds", "attention_mask"], output_names=["hidden"],
            dynamic_axes={"inputs_embeds": {0: "batch", 1: "seq"},
                          "attention_mask": {0: "batch", 1: "seq"},
                          "hidden": {0: "batch", 1: "seq"}},
            opset_version=opset, dynamo=False,
        )
        past = torch.zeros(layers, 1, heads, 2, d_kv)
        torch.onnx.export(
            step, (emb[torch.tensor([[5]])], hidden, mask, past, past),
            str(out_dir / "decoder.onnx"),
            input_names=["inputs_embeds", "encoder_hidden_states", "encoder_attention_mask",
                         "past_keys", "past_values"],
            output_names=["logits", "present_keys", "present_values"],
            dynamic_axes={"inputs_embeds": {0: "batch"},
                          "encoder_hidden_states": {0: "batch", 1: "seq"},
                          "encoder_attention_mask": {0: "batch", 1: "seq"},
                          "past_keys": {1: "batch", 3: "past"},
                          "past_values": {1: "batch", 3: "past"},
                          "logits": {0: "batch"},
                          "present_keys": {1: "batch", 3: "total"},
                          "present_values": {1: "batch", 3: "total"}},
            opset_version=opset, dynamo=False,
        )

    # Parity: step-by-step ONNX decoding (starting from an empty cache) vs the full model.
    enc_s = ort.InferenceSession(str(out_dir / "encoder.onnx"), providers=["CPUExecutionProvider"])
    dec_s = ort.InferenceSession(str(out_dir / "decoder.onnx"), providers=["CPUExecutionProvider"])
    h = enc_s.run(None, {"inputs_embeds": emb[ids].numpy(), "attention_mask": mask.numpy()})[0]
    enc_diff = float(np.abs(h - hidden.numpy()).max())
    pk = np.zeros((layers, 1, heads, 0, d_kv), dtype=np.float32)
    pv = pk.copy()
    dec_diff = 0.0
    for t, tok in enumerate(seq):
        logits, pk, pv = dec_s.run(None, {
            "inputs_embeds": emb[torch.tensor([[tok]])].numpy(),
            "encoder_hidden_states": h, "encoder_attention_mask": mask.numpy(),
            "past_keys": pk, "past_values": pv})
        dec_diff = max(dec_diff, float(np.abs(logits[0] - ref[t]).max()))
    stats = {"encoder_max_abs_diff": enc_diff, "decoder_logits_max_abs_diff": dec_diff,
             "steps_checked": len(seq), "output_scale": step.scale}
    if enc_diff > tolerance or dec_diff > tolerance:
        raise RuntimeError(f"ONNX export does not match the model: {stats}")
    return stats

In [ ]:
# Dataset construction parameters (dataset shape, not acceptance thresholds).
LEVELS = ['ANY', 'ENTRY', 'MID', 'SENIOR', 'LEAD']  # == findrr SeniorityLevel + ANY
LEVEL_PATTERNS = {
    'ENTRY': r'\b(junior|jr|entry|associate|trainee|apprentice|intern|graduate)\b',
    'SENIOR': r'\b(senior|sr)\b',
    'LEAD': r'\b(lead|principal|chief|head)\b',
}
ANY_LEVEL = re.compile('|'.join(LEVEL_PATTERNS.values()), re.IGNORECASE)
MAX_ANCHORS_PER_OCCUPATION = 6
# D-010: extra training-only inputs from O*NET job titles, level ANY.
MAX_EXTRA_ANCHORS_PER_OCCUPATION = 30
MAX_TARGET_TITLES = 5
MIN_TARGET_TITLES = 3
MAX_SKILLS = 5

def level_pool(level, neutral, all_titles):
    if level in LEVEL_PATTERNS:
        pat = re.compile(LEVEL_PATTERNS[level], re.IGNORECASE)
        pool = sorted({t for t in all_titles if pat.search(t)}, key=str.lower)
        if len(pool) >= MIN_TARGET_TITLES:
            return pool
    return neutral

def skills_text(skill_rows, rng):
    ranked = [s for s, hot in skill_rows if hot] + [s for s, hot in skill_rows if not hot]
    ranked = list(dict.fromkeys(ranked))[:8]
    if not ranked:
        return 'general'  # what the Findrr runner sends when no skills exist
    k = rng.randint(1, min(MAX_SKILLS, len(ranked)))
    return ', '.join(ranked[:k])

def occupation_anchors(onet):
    """{soc: (anchors, neutral titles, all titles)} for occupations with enough targets."""
    out = {}
    for soc, occ_title in sorted(onet['occupations'].items()):
        # ',' separates titles in the model output, so titles containing commas
        # (e.g. 'Superintendent, Schools') are left out of inputs and targets.
        base = [occ_title] if ',' not in occ_title else []
        reported = [t for t in onet['reported'].get(soc, []) if ',' not in t]
        all_titles = [t for t in dict.fromkeys(base + reported + onet['job_titles'].get(soc, []))
                      if ',' not in t]
        neutral = sorted({t for t in base + reported if not ANY_LEVEL.search(t)}, key=str.lower)
        if len(neutral) < MIN_TARGET_TITLES + 1:
            continue
        anchors = base + [t for t in neutral if not base or norm(t) != norm(occ_title)]
        out[soc] = (anchors[:MAX_ANCHORS_PER_OCCUPATION], neutral, all_titles)
    return out

def _anchor_hash(seed, key):
    import hashlib
    return int(hashlib.sha256(f'{seed}:{key}'.encode()).hexdigest()[:12], 16) / float(16 ** 12)

def split_by_anchor(anchors_by_soc, seed=42, fractions=(0.8, 0.1, 0.1)):
    """D-009: assign each normalised input title (anchor) to train/val/test.

    Every occupation is learned, but evaluation inputs are wordings never used as a
    training input: the same anchor text never crosses splits, in any occupation.
    Each occupation keeps at least one training anchor.
    """
    def bucket(key):
        h = _anchor_hash(seed, key)
        return 'train' if h < fractions[0] else 'val' if h < fractions[0] + fractions[1] else 'test'
    split = {}
    for anchors in anchors_by_soc.values():
        for a in anchors:
            split.setdefault(norm(a), bucket(norm(a)))
    for soc in sorted(anchors_by_soc):
        keys = [norm(a) for a in anchors_by_soc[soc]]
        if keys and not any(split[k] == 'train' for k in keys):
            split[min(keys, key=lambda k: _anchor_hash(seed, k))] = 'train'
    return split

def build_title_examples(onet, seed=42, split_of=None):
    """Return {split: [{'input_text', 'target_text', 'soc', 'anchor'}]} in the runner's
    prompt format. ``split_of`` maps normalised anchors to splits (default: split_by_anchor)."""
    rng = random.Random(seed)
    anchors_by_soc = occupation_anchors(onet)
    if split_of is None:
        split_of = split_by_anchor({s: a for s, (a, _, _) in anchors_by_soc.items()}, seed=seed)
    out = {'train': [], 'val': [], 'test': []}
    for soc, (anchors, neutral, all_titles) in anchors_by_soc.items():
        for level in LEVELS:
            pool = level_pool(level, neutral, all_titles)
            for anchor in anchors:
                targets = [t for t in pool if norm(t) != norm(anchor)][:MAX_TARGET_TITLES]
                if len(targets) < MIN_TARGET_TITLES:
                    continue
                out[split_of[norm(anchor)]].append({
                    'input_text': f'expand titles: {anchor} | skills: '
                                  f'{skills_text(onet["skills"].get(soc, []), rng)} | level: {level}',
                    'target_text': ', '.join(targets),
                    'soc': soc,
                    'anchor': anchor,
                })
        out['train'] += extra_anchor_rows(onet, soc, neutral, all_titles, split_of, seed, rng)
    return out

def extra_anchor_rows(onet, soc, neutral, all_titles, split_of, seed, rng):
    """D-010: more ways to say the same job, for training only.

    Job titles that are not already anchors, carry no level word, and are not the
    wording of any val/test anchor (in any occupation) become level-ANY inputs whose
    targets are the occupation's neutral titles.
    """
    extra = [t for t in all_titles
             if norm(t) not in split_of and not ANY_LEVEL.search(t)]
    extra = sorted(dict.fromkeys(extra), key=lambda t: _anchor_hash(seed, norm(t)))
    rows = []
    for anchor in extra[:MAX_EXTRA_ANCHORS_PER_OCCUPATION]:
        targets = [t for t in neutral if norm(t) != norm(anchor)][:MAX_TARGET_TITLES]
        if len(targets) < MIN_TARGET_TITLES:
            continue
        rows.append({
            'input_text': f'expand titles: {anchor} | skills: '
                          f'{skills_text(onet["skills"].get(soc, []), rng)} | level: ANY',
            'target_text': ', '.join(targets),
            'soc': soc,
            'anchor': anchor,
        })
    return rows

def novel_wording(examples):
    """Normalised test anchors never seen in training, as an input or as a target."""
    seen = {norm(r['anchor']) for r in examples['train']}
    seen |= {norm(t) for r in examples['train'] for t in parse_titles(r['target_text'])}
    return {norm(r['anchor']) for r in examples['test']} - seen

def title_dictionary(onet):
    """Every known O*NET title, normalised: the 'approved occupational dictionary'."""
    titles = set()
    for soc, t in onet['occupations'].items():
        titles.add(norm(t))
    for table in (onet['reported'], onet['job_titles']):
        for lst in table.values():
            titles.update(norm(t) for t in lst)
    return titles

def dictionary_titles(onet):
    """Comma-free O*NET titles: the constrained-decoding vocabulary (titles.json)."""
    titles = [t for t in onet['occupations'].values() if ',' not in t]
    for table in (onet['reported'], onet['job_titles']):
        for lst in table.values():
            titles += [t for t in lst if ',' not in t]
    return sorted(dict.fromkeys(titles), key=str.lower)

def title_occupations(onet):
    """Normalised title -> set of O*NET-SOC codes that list it."""
    index = defaultdict(set)
    for soc, t in onet['occupations'].items():
        index[norm(t)].add(soc)
    for table in (onet['reported'], onet['job_titles']):
        for soc, lst in table.items():
            for t in lst:
                index[norm(t)].add(soc)
    return index

def occupation_precision(soc_and_titles, onet, index):
    """Share of generated titles belonging to the input occupation or to an O*NET
    'Primary-Short' related occupation (D-007 metric, replaces ROUGE-L)."""
    hit = total = 0
    for soc, titles in soc_and_titles:
        ok = {soc} | {r for r, tier in onet['related'].get(soc, []) if tier == 'Primary-Short'}
        for t in titles:
            total += 1
            hit += bool(index.get(norm(t), set()) & ok)
    return hit / total if total else 0.0

def parse_titles(text):
    return [t.strip() for t in text.split(',') if t.strip()]

def valid_title_ratio(predictions, dictionary):
    titles = [t for p in predictions for t in parse_titles(p)]
    return sum(norm(t) in dictionary for t in titles) / len(titles) if titles else 0.0

def repetition_rate(predictions):
    rates = []
    for p in predictions:
        ts = [norm(t) for t in parse_titles(p)]
        rates.append((len(ts) - len(set(ts))) / len(ts) if ts else 0.0)
    return sum(rates) / len(rates) if rates else 0.0

In [ ]:
onet = load_onet(ONET_DIR)
examples = build_title_examples(onet, seed=SEED)
DICTIONARY = title_dictionary(onet)
for name, rows in examples.items():
    print(f'{name}: {len(rows)} examples from {len({r["soc"] for r in rows})} occupations')
# D-009: inputs are disjoint across splits; occupations are shared on purpose.
overlap = {norm(r['anchor']) for r in examples['train']} & {norm(r['anchor']) for r in examples['test']}
assert not overlap, 'input-title leakage between train and test'
assert {r['soc'] for r in examples['test']} <= {r['soc'] for r in examples['train']}
NOVEL = novel_wording(examples)
print(f'Test anchors never seen in training (input or target): {len(NOVEL)}')
print('Sample:', examples['train'][0])

In [ ]:
import importlib
def _check(module, expected):
    found = importlib.import_module(module).__version__
    if found != expected:
        raise RuntimeError(
            f'{module} {found} is loaded but {expected} is pinned. The kernel imported it '
            'before the pip install finished. Use Run > Restart & clear cell outputs, then '
            'Run All again.')
_check('transformers', '4.46.3')
_check('datasets', '3.1.0')

In [ ]:
import json, shutil
import numpy as np, torch
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments, set_seed)
set_seed(SEED)
BASE_MODEL = 'google/flan-t5-small'
MAX_INPUT_LEN, MAX_TARGET_LEN = 128, 64
# D-004: to skip training and re-export an accepted v2.0.0 model, upload its bundle
# as ONE archive (title_expander_bundle.tar.gz or a .zip) to a private Kaggle dataset,
# attach it, and set this to the dataset folder, e.g. '/kaggle/input/findrr-title-expander-v200'.
REUSE_BUNDLE_DIR = None
REUSED_BUNDLE = None
if REUSE_BUNDLE_DIR:
    import hashlib, tarfile, zipfile
    src = Path(REUSE_BUNDLE_DIR)
    hits = list(src.rglob('manifest.json')) if src.is_dir() else []
    if not hits:
        # Kaggle datasets hold files, not folders: accept the bundle as one archive
        # (.tar.gz or .zip), whether or not Kaggle unpacked it on upload.
        archives = [src] if src.is_file() else sorted(
            p for p in src.rglob('*') if p.name.endswith(('.tar.gz', '.tgz', '.zip')))
        if len(archives) != 1:
            raise RuntimeError(f'Expected one bundle archive under {src}, found {len(archives)}.')
        unpacked = Path('/kaggle/working/reused-bundle')
        shutil.rmtree(unpacked, ignore_errors=True); unpacked.mkdir(parents=True)
        if archives[0].name.endswith('.zip'):
            with zipfile.ZipFile(archives[0]) as zf:
                names = zf.namelist()
                if any(n.startswith('/') or '..' in Path(n).parts for n in names):
                    raise RuntimeError('Unsafe path in the bundle archive.')
                zf.extractall(unpacked)
        else:
            with tarfile.open(archives[0]) as tar:
                members = tar.getmembers()
                if any(not (m.isfile() or m.isdir()) or m.name.startswith('/')
                       or '..' in Path(m.name).parts for m in members):
                    raise RuntimeError('Unsafe entry in the bundle archive.')
                tar.extractall(unpacked, members=members)
        hits = list(unpacked.rglob('manifest.json'))
    if len(hits) != 1:
        raise RuntimeError(f'Expected exactly one manifest.json under {src}, found {len(hits)}.')
    src = hits[0].parent
    src_manifest = json.loads(hits[0].read_text())
    if (src_manifest.get('model_id'), src_manifest.get('version')) != ('ML-001', '2.0.0'):
        raise RuntimeError(f'Not an ML-001 v2.0.0 bundle: {src_manifest.get("model_id")} {src_manifest.get("version")}')
    if not src_manifest['metadata']['acceptance']['passed']:
        raise RuntimeError('The reused bundle did not pass acceptance; refusing to re-export it.')
    for name, digest in src_manifest['files'].items():
        if hashlib.sha256((src / name).read_bytes()).hexdigest() != digest:
            raise RuntimeError(f'Checksum mismatch in the reused bundle: {name}')
    REUSED_BUNDLE = {'version': src_manifest['version'], 'export_date': src_manifest['export_date'],
                     'manifest_sha256': hashlib.sha256(hits[0].read_bytes()).hexdigest()}
    print('Reusing verified bundle (no training):', REUSED_BUNDLE)
    tokenizer = AutoTokenizer.from_pretrained(str(src))
    model = AutoModelForSeq2SeqLM.from_pretrained(str(src))
else:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

if REUSED_BUNDLE is None:
    from datasets import Dataset

    def tokenize(batch):
        enc = tokenizer(batch['input_text'], max_length=MAX_INPUT_LEN, truncation=True)
        enc['labels'] = tokenizer(text_target=batch['target_text'], max_length=MAX_TARGET_LEN,
                                  truncation=True)['input_ids']
        return enc

    ds = {k: Dataset.from_list(v).map(tokenize, batched=True, remove_columns=['input_text', 'target_text', 'soc', 'anchor'])
          for k, v in examples.items()}

    # FP16 is off: T5 checkpoints are known to overflow in fp16 and the T4 has no bf16.
    # flan-t5-small trains comfortably in fp32 on a T4.
    args = Seq2SeqTrainingArguments(
        output_dir='/kaggle/working/ml001-checkpoints',
        learning_rate=3e-4, lr_scheduler_type='cosine', warmup_ratio=0.1,
        num_train_epochs=15, per_device_train_batch_size=32, per_device_eval_batch_size=64,
        weight_decay=0.01, eval_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True, metric_for_best_model='eval_loss', save_total_limit=1,
        fp16=False, logging_steps=50, seed=SEED, report_to='none',
    )
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=ds['train'], eval_dataset=ds['val'],
                             data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
                             processing_class=tokenizer)
    trainer.train()
    model = trainer.model  # best checkpoint (load_best_model_at_end)

## Export to ONNX (D-004), then evaluate the exported files

The export writes `encoder.onnx`, a KV-cached one-step `decoder.onnx` and
`embeddings.npy`, and stops if the ONNX logits differ from the model's own
logits. Evaluation then uses only those files and the app's decoding code.

In [ ]:
import shutil
EXPORT_DIR = Path('/kaggle/working/title-expander-v2.1.0')
shutil.rmtree(EXPORT_DIR, ignore_errors=True); EXPORT_DIR.mkdir(parents=True)
ONNX_PARITY = export_t5_bundle(model, EXPORT_DIR)
print('ONNX parity:', ONNX_PARITY)
model.config.save_pretrained(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)
for leftover in ('spiece.model', 'generation_config.json'):
    (EXPORT_DIR / leftover).unlink(missing_ok=True)  # not used by the ONNX runner

In [ ]:
from rouge_score import rouge_scorer
import sacrebleu

MAX_NEW_TOKENS = 64  # == MAX_TARGET_LEN, so 5 titles are not cut off
NUM_BEAMS = 2
# D-010: 1.0 (off). The constraint already prevents repeated titles, and a token
# penalty works against correct sibling titles, which share words.
REPETITION_PENALTY = 1.0
# The app's own inference path: ONNX Runtime + tokenizers, no torch (D-004).
onnx_tokenizer = TokenizersAdapter(EXPORT_DIR / 'tokenizer.json', max_length=MAX_INPUT_LEN)
onnx_generator = T5OnnxGenerator(EXPORT_DIR)
# D-007: decoding is constrained to real O*NET titles; repeats cannot be completed.
DICTIONARY_TITLES = dictionary_titles(onet)
constraint = TitleTrieConstraint(DICTIONARY_TITLES, onnx_tokenizer, max_titles=MAX_TARGET_TITLES)
print(f'Constrained vocabulary: {len(constraint.titles)} titles ({len(constraint.skipped)} skipped)')

def generate(texts):
    """Returns (raw decoded text, complete distinct titles) per input, as the runner does."""
    raw, titles = [], []
    for text in texts:
        out = onnx_generator.generate(onnx_tokenizer(text)['input_ids'], constraint.allowed_tokens,
                                      num_beams=NUM_BEAMS, max_new_tokens=MAX_NEW_TOKENS)
        raw.append(onnx_tokenizer.decode(out, skip_special_tokens=True))
        titles.append(constraint.complete_titles(out, onnx_tokenizer))
    return raw, titles

test_rows = examples['test']
raw_preds, title_preds = generate([r['input_text'] for r in test_rows])
test_targets = [r['target_text'] for r in test_rows]
INDEX = title_occupations(onet)
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
measured = {
    'occupation_precision': occupation_precision(
        [(r['soc'], t) for r, t in zip(test_rows, title_preds)], onet, INDEX),
    # Measured on the raw text, so a title cut off by max length counts as invalid.
    'valid_title_ratio': valid_title_ratio(raw_preds, DICTIONARY),
    'repetition_rate': repetition_rate(raw_preds),
    'mean_distinct_titles': float(np.mean([len(t) for t in title_preds])),
    # Reported only (D-009): inputs whose wording never appeared in training at all.
    'occupation_precision_novel_wording': occupation_precision(
        [(r['soc'], t) for r, t in zip(test_rows, title_preds) if norm(r['anchor']) in NOVEL],
        onet, INDEX),
    'novel_wording_examples': float(sum(norm(r['anchor']) in NOVEL for r in test_rows)),
    # Reported only (D-007): agreement with one arbitrary 5-title reference.
    'rougeL': float(np.mean([scorer.score(t, p)['rougeL'].fmeasure for t, p in zip(test_targets, raw_preds)])),
    'bleu4': float(sacrebleu.corpus_bleu(raw_preds, [test_targets]).score / 100.0),
    'test_examples': float(len(raw_preds)),
}
# Reported only (D-010): the same metric on seeded training inputs, to show under-fitting.
train_sample = random.Random(SEED).sample(examples['train'], min(500, len(examples['train'])))
_, train_titles = generate([r['input_text'] for r in train_sample])
measured['occupation_precision_train_sample'] = occupation_precision(
    [(r['soc'], t) for r, t in zip(train_sample, train_titles)], onet, INDEX)
demo_raw, demo_titles = generate(['expand titles: Software Developer | skills: Python, SQL, Git | level: ANY'])
measured['ac_ml_001_1_distinct_titles'] = float(len(demo_titles[0]))
print('Software Developer ->', demo_titles[0])
for r, t in list(zip(test_rows, title_preds))[:15]:
    print(f"  {r['anchor']!r:40} -> {t}")
print(json.dumps(measured, indent=2))

In [ ]:
import hashlib, json
from datetime import datetime, timezone
from pathlib import Path

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        while chunk := fh.read(65536):
            h.update(chunk)
    return h.hexdigest()

def build_manifest(export_dir, model_id, model_name, version, architecture,
                   evaluation_summary, acceptance, metadata) -> dict:
    """Manifest in the exact schema of findrr.ml.manifest.ModelManifest.

    ``files`` maps every exported file (except manifest.json) to its SHA-256.
    ``evaluation_summary`` holds measured floats only; nothing is defaulted.
    """
    export_dir = Path(export_dir)
    files = {
        p.name: sha256_file(p)
        for p in sorted(export_dir.iterdir())
        if p.is_file() and p.name != 'manifest.json'
    }
    for key, value in evaluation_summary.items():
        if not isinstance(value, (int, float)) or value != value:  # reject None/NaN
            raise ValueError(f'Metric {key!r} was not measured: {value!r}')
    return {
        'model_id': model_id,
        'model_name': model_name,
        'version': version,
        'architecture': architecture,
        'export_date': datetime.now(timezone.utc).isoformat(),
        'files': files,
        'evaluation_summary': {k: float(v) for k, v in evaluation_summary.items()},
        'metadata': {**metadata, 'acceptance': acceptance},
    }

def check_acceptance(measured: dict, criteria: dict) -> dict:
    """criteria: {metric: ('>=' | '<', threshold)} taken verbatim from the model spec."""
    results = {}
    for metric, (op, threshold) in criteria.items():
        value = measured[metric]
        ok = value >= threshold if op == '>=' else value < threshold
        results[metric] = {'value': float(value), 'op': op, 'threshold': threshold, 'passed': bool(ok)}
    return {'passed': all(r['passed'] for r in results.values()), 'criteria': results}

In [ ]:
# Criteria from docs/ml/models/title-expander.md as amended by D-007 (option 1).
# OCCUPATION_PRECISION_THRESHOLD was approved by the owner (PR #27). D-009 moved the
# split to held-out input titles, so every occupation's titles are learnable.
# Reference points (O*NET 31.0): random dictionary title ~0.007, same SOC major
# group ~0.12, same SOC broad group (e.g. 15-125x) ~0.74, gold references 1.0.
OCCUPATION_PRECISION_THRESHOLD = 0.75
ACCEPTANCE = check_acceptance(measured, {
    'occupation_precision': ('>=', OCCUPATION_PRECISION_THRESHOLD),
    'valid_title_ratio': ('>=', 0.85),
    'repetition_rate': ('<', 0.05),
    'ac_ml_001_1_distinct_titles': ('>=', 3),
})

In [ ]:
# Acceptance gate. A model that fails the approved criteria is NOT exported unless you
# deliberately set ALLOW_EXPORT_BELOW_ACCEPTANCE = True. In that case the manifest records
# acceptance.passed = false and the Findrr app refuses to load the model.
ALLOW_EXPORT_BELOW_ACCEPTANCE = False
print(json.dumps(ACCEPTANCE, indent=2))
if not ACCEPTANCE['passed'] and not ALLOW_EXPORT_BELOW_ACCEPTANCE:
    raise RuntimeError(
        'ML-001 did not meet its acceptance criteria. Not exporting. '
        'Report the metrics above; do not change thresholds without approval.'
    )

In [ ]:
import tarfile
(EXPORT_DIR / 'preprocess_config.json').write_text(json.dumps({
    'max_input_length': MAX_INPUT_LEN, 'max_target_length': MAX_TARGET_LEN,
    'input_template': 'expand titles: {title} | skills: {skills} | level: {level}',
    'levels': LEVELS, 'num_beams': NUM_BEAMS, 'do_sample': False,
    'repetition_penalty': REPETITION_PENALTY, 'max_new_tokens': MAX_NEW_TOKENS,
    'constrained_decoding': True, 'max_titles': MAX_TARGET_TITLES,
}, indent=2))
# Constrained-decoding dictionary, read by findrr TitleExpanderRunner (D-007).
(EXPORT_DIR / 'titles.json').write_text(json.dumps(constraint.titles, ensure_ascii=False))
import transformers, onnxruntime as ort
manifest = build_manifest(
    EXPORT_DIR, 'ML-001', 'title-expander', '2.1.0', 'google/flan-t5-small (ONNX fp32, KV-cached decoder)',
    evaluation_summary=measured, acceptance=ACCEPTANCE,
    metadata={'dataset': f'O*NET {ONET_VERSION} (CC BY 4.0)', 'dataset_sha256': ONET_SHA256,
              'split': 'by input title (anchor), hashed 80/10/10; every occupation in training (D-009)',
              'examples': {k: len(v) for k, v in examples.items()}, 'seed': SEED,
              'transformers': transformers.__version__, 'torch': torch.__version__,
              'onnxruntime': ort.__version__, 'onnx_parity': ONNX_PARITY,
              'reused_bundle': REUSED_BUNDLE},
)
(EXPORT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2))
BUNDLE = '/kaggle/working/title_expander_bundle.tar.gz'
with tarfile.open(BUNDLE, 'w:gz') as tar:
    for p in sorted(EXPORT_DIR.iterdir()):
        tar.add(p, arcname=p.name)
print(json.dumps(manifest, indent=2))

In [ ]:
# Re-verify the bundle exactly as the Findrr app will (SHA-256 of every file).
import tarfile, tempfile
with tempfile.TemporaryDirectory() as tmp:
    with tarfile.open('/kaggle/working/title_expander_bundle.tar.gz') as tar:
        tar.extractall(tmp)
    m = json.loads((Path(tmp) / 'manifest.json').read_text())
    for name, digest in m['files'].items():
        assert sha256_file(Path(tmp) / name) == digest, name
print('[OK] Bundle verified:', len(m['files']), 'files;', 'acceptance passed =', m['metadata']['acceptance']['passed'])